# 16 — Confronto Finale per la Tesi: FCGR Baseline vs FCGR Ottimizzate

## Scopo del Notebook
Raccogliere e confrontare tutte le metriche sperimentali ottenute tra la **FCGR Baseline ($k=6$)** e le **Varianti FCGR Ottimizzate** (che implementeremo in `src/fcgr.py` nella Fase 2):
1. **Efficienza Computazionale**: Tempo di conversione (`ms`/sequenza), Speedup (`x`), Memory Footprint (`KB`/`MB`/`GB`) e Indice di Sparsità ($S\%$).
2. **Validazione Downstream Binaria (Hotspot vs Stabile) sulla Rete Siamese (`SiameseV3`)**:
   - **ROC-AUC** e **PR-AUC** (sia Pairwise che Classificazione Binaria);
   - **F1-Score, Precision, Recall, Balanced Accuracy**;
   - **Separabilità degli Embedding ($d^+$, $d^-$, Gap $\Delta d$, $d'$)**.

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('META_PATH    :', META_PATH, '-> exists:', META_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

from src.fcgr import (
    compute_fcgr,
    normalize_fcgr,
    plot_fcgr_diagnostics,
    plot_fcgr_comparison,
)

ORIGINAL_RESULTS_DIR = PROJECT_ROOT / 'original_results' / 'fcgr_analysis'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

## 1. Analisi del Collo di Bottiglia della FCGR Baseline (Sparsità & Memoria)
Carichiamo i risultati generati dal Notebook `02` (o dai CSV di partenza in `original_results/fcgr_analysis/`) per documentare la crescita esponenziale della memoria e della sparsità.

In [ ]:
k_summary_path = RESULTS_DIR / 'final_k_summary.csv'
if not k_summary_path.exists():
    k_summary_path = ORIGINAL_RESULTS_DIR / 'final_k_summary.csv'

baseline_k_df = pd.read_csv(k_summary_path)
display(baseline_k_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(baseline_k_df['k'], baseline_k_df['median_sparsity'], marker='o', linewidth=2, color='#C53030')
axes[0].axhline(90.0, linestyle='--', color='gray', label='90% Sparsità')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Sparsità Mediana (%)')
axes[0].set_title('Sparsità della FCGR Baseline al crescere di k')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].semilogy(baseline_k_df['k'], baseline_k_df['train_dense_float32_GB'], marker='s', linewidth=2, label='Train Set (GB)')
axes[1].semilogy(baseline_k_df['k'], baseline_k_df['full_manifest_dense_float32_GB'], marker='^', linewidth=2, label='Full Manifest (GB)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Memoria Densa float32 (GB)')
axes[1].set_title('Crescita Esponenziale del Memory Footprint ($4^k$)')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'baseline_bottleneck_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Confronto Metriche Downstream sulla Rete Siamese Binaria (`04_siamese_model.ipynb`)
Carica automaticamente le metriche salvate dal run della Baseline (`artifacts/siamese_baseline_k6/baseline_final_metrics.csv`) e le mette a confronto con le varianti FCGR ottimizzate.

In [ ]:
metric_files = sorted(ARTIFACTS_DIR.glob('siamese_*/baseline_final_metrics.csv')) + sorted(ARTIFACTS_DIR.glob('siamese_opt_*/*_final_metrics.csv'))

if len(metric_files) > 0:
    dfs = []
    for f in metric_files:
        df_m = pd.read_csv(f)
        df_m.insert(0, 'experiment', f.parent.name)
        dfs.append(df_m)
    comparison_table = pd.concat(dfs, ignore_index=True)
    comparison_table.to_csv(COMPARISON_DIR / 'siamese_binary_comparison_table.csv', index=False)
    display(comparison_table)
else:
    print('Nessun file baseline_final_metrics.csv ancora trovato in artifacts/. Esegui prima il Notebook 04 su Colab!')

## 3. Confronto Visivo Diretto: FCGR Baseline vs FCGR Ottimizzata (`plot_fcgr_comparison`)

In [ ]:
demo_seq = ('ACGTACGTACGTGCGCATATGCGCATCGTAGCTAGCTAGCATCGATCG' * 12)
t0 = time.perf_counter()
c_base, _, _ = compute_fcgr(demo_seq, k=6)
f_base = normalize_fcgr(c_base)
t_base_ms = (time.perf_counter() - t0) * 1000.0

_ = plot_fcgr_comparison(
    fcgr_baseline=f_base,
    fcgr_optimized=f_base,
    time_baseline_ms=t_base_ms,
    time_optimized_ms=t_base_ms,
    name_baseline='FCGR Baseline (k=6, float32)',
    name_optimized='FCGR Ottimizzata (Fase 2)',
    save_path=COMPARISON_DIR / 'fcgr_visual_comparison.png',
)